# spaCy: Processamento de Linguagem Natural em Português

O [spaCy](https://spacy.io/) é uma biblioteca industrial de PLN rápida, robusta e com
modelos pré-treinados para dezenas de idiomas — incluindo o português. Diferente do
NLTK (que veremos no próximo capítulo), o spaCy foi projetado desde o início para
produção: pipelines otimizados, vetores de palavras integrados e uma API orientada a
objetos que transforma texto bruto em anotações linguísticas com poucas linhas.

**O que este notebook cobre:**

- Carregar o modelo estatístico `pt_core_news_lg` (500 MB+, vetores de 300d)
- Tokens, atributos lexicais e sentenças
- Classes gramaticais (POS tagging), lematização e morfologia
- Dependências sintáticas
- Reconhecimento de Entidades Nomeadas (NER)
- Stopwords e vocabulário
- Vetores de palavras e similaridade semântica
- Casamento de padrões com `Matcher`
- Visualização com `displaCy`
- Gerenciamento do pipeline

**Material de origem:** Curso [Formação Processamento de Linguagem Natural, LLMs e
GenAI](https://www.udemy.com/course/formacao-processamento-de-linguagem-natural-nlp/)
— Prof. Fernando Amaral.

## 1. Instalação e carregamento do modelo

O modelo `pt_core_news_lg` é o pacote grande para português. O nome decodifica como:

| Componente | Significado |
|---|---|
| `pt` | idioma Português |
| `core` | pipeline completo (tokenização, POS, parser, NER, lematizador) |
| `news` | treinado sobre textos jornalísticos e artigos da web |
| `lg` | versão *large* (~500 MB) com vetores de palavras de 300 dimensões |

O download é feito uma única vez:

In [ ]:
#!python -m spacy download pt_core_news_lg

In [ ]:
import spacy

nlp = spacy.load('pt_core_news_lg')
print(type(nlp))

## 2. O pipeline de processamento

Quando chamamos `nlp(texto)`, o spaCy executa uma sequência de componentes em ordem.
Cada componente adiciona anotações ao documento. O pipeline do `pt_core_news_lg` é:

In [ ]:
print(nlp.pipe_names)

| Componente | Função |
|---|---|
| `tok2vec` | codifica tokens em vetores densos (base para os demais) |
| `morphologizer` | prediz classe gramatical (POS) e traços morfológicos |
| `parser` | monta a árvore de dependências sintáticas |
| `lemmatizer` | reduz cada token à sua forma canônica (lema) |
| `attribute_ruler` | ajusta atributos com regras manuais |
| `ner` | reconhece entidades nomeadas (pessoas, locais, organizações) |

A ordem importa: o parser usa o POS, e o NER usa o parser. Remover um componente
pode quebrar os seguintes.

## 3. Processando texto — o objeto `Doc`

O coração do spaCy é o objeto `Doc`: uma sequência imutável de tokens anotados.
Vamos processar uma sentença de exemplo e inspecionar sua estrutura.

In [ ]:
documento = nlp(
    "As ações do Magazine Luiza S.A., Franca, Brasil, "
    "acumularam baixa de 70% ao ano. "
    "Assim já devolveram todos os ganhos do período da pandemia."
)

print(type(documento))
print(f"Tokens: {len(documento)}")
print(f"Vocabulário carregado: {len(documento.vocab):,} entradas")

## 4. Tokenização

A tokenização é o primeiro passo: o texto é segmentado em palavras, pontuação,
números e símbolos. O spaCy faz isso com regras linguísticas específicas do idioma.

In [ ]:
# Iterar sobre todos os tokens
for token in documento:
    print(token.text)

In [ ]:
# Acessar token por índice
print(documento[3])       # 'Magazine'

In [ ]:
# Span (fatia do documento)
print(documento[3:5])     # 'Magazine Luiza'

### Atributos booleanos dos tokens

Cada token expõe dezenas de atributos. Os mais usados na prática:

In [ ]:
print(f"{'texto':<15} {'stop':>6} {'alpha':>6} {'upper':>6} {'punct':>6} {'num':>6} {'sent_start':>11}")
print("-" * 65)
for token in documento:
    print(
        f"{token.text:<15} "
        f"{str(token.is_stop):>6} "
        f"{str(token.is_alpha):>6} "
        f"{str(token.is_upper):>6} "
        f"{str(token.is_punct):>6} "
        f"{str(token.like_num):>6} "
        f"{str(token.is_sent_start):>11}"
    )

| Atributo | Tipo | Significado |
|---|---|---|
| `token.is_stop` | `bool` | Palavra de parada (artigo, preposição, conjunção...) |
| `token.is_alpha` | `bool` | Composto apenas por letras |
| `token.is_upper` | `bool` | Todo em maiúsculas |
| `token.is_punct` | `bool` | Sinal de pontuação |
| `token.like_num` | `bool` | Representa um número (dígito ou por extenso) |
| `token.is_sent_start` | `bool` | Início de sentença |
| `token.shape_` | `str` | Esquema ortográfico (`Xxxx` = maiúscula+minúsculas, `dd` = dois dígitos) |

In [ ]:
# Filtrar por tipo
for token in documento:
    if token.like_num:
        print(f"Número: {token.text}")
    if token.is_punct:
        print(f"Pontuação: {token.text}")

In [ ]:
# Formato esquemático (shape_)
for token in documento:
    print(f"{token.text:<15} shape={token.shape_}")

## 5. Classes gramaticais (POS tagging)

O spaCy atribui a cada token uma etiqueta de classe gramatical.
As duas principais são:

- **`pos_`** — etiqueta universal (*Universal Dependencies*): `NOUN`, `VERB`, `ADJ`, `DET`, ...
- **`tag_`** — etiqueta granular específica do idioma

In [ ]:
print(f"{'texto':<15} {'pos_':<8} {'tag_':<8} {'descrição'}")
print("-" * 55)
for token in documento:
    print(f"{token.text:<15} {token.pos_:<8} {token.tag_:<8} {spacy.explain(token.tag_)}")

`spacy.explain()` é uma função utilitária que traduz qualquer etiqueta do spaCy
em uma descrição legível — útil para debugging e aprendizado.

In [ ]:
# Exemplos rápidos
print(spacy.explain('PROPN'))   # proper noun
print(spacy.explain('AUX'))     # auxiliary verb
print(spacy.explain('ADP'))     # adposition (preposição)

## 6. Morfologia

Além da classe gramatical, o spaCy extrai traços morfológicos finos: gênero,
número, pessoa, tempo verbal, modo...

In [ ]:
for token in documento:
    if token.pos_ in ('NOUN', 'VERB', 'DET', 'PROPN'):
        print(f"{token.text:<15} {str(token.morph):<50} pos={token.pos_}")

Cada traço pode ser acessado individualmente via `token.morph.get('Gender')`,
`token.morph.get('Number')` etc.

## 7. Lematização

O **lema** é a forma canônica da palavra: `acumularam` → `acumular`,
`ações` → `ação`. É essencial para normalização em tarefas de busca, indexação e
análise semântica — diferente do *stemming* (que trunca radicalmente), a
lematização preserva a classe gramatical.

In [ ]:
print(f"{'texto':<15} {'lema':<15} {'pos_':<8}")
print("-" * 40)
for token in documento:
    print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<8}")

## 8. Dependências sintáticas

O parser constrói uma árvore de relações gramaticais. Cada token tem:

- **`dep_`** — tipo da relação (`nsubj`, `obj`, `det`, `ROOT`, ...)
- **`head`** — token pai na árvore (o `ROOT` é o verbo principal)

In [ ]:
print(f"{'texto':<15} {'dep_':<12} {'head.text':<12} {'pos_'}")
print("-" * 55)
for token in documento:
    print(f"{token.text:<15} {token.dep_:<12} {token.head.text:<12} {token.pos_}")

Interpretação: `ações` é sujeito (`nsubj`) do verbo `acumularam`; `Magazine` e
`Luiza` modificam `ações`; `baixa` é objeto direto (`obj`) de `acumularam`.

## 9. Entidades Nomeadas (NER)

O reconhecedor de entidades identifica organizações, locais, pessoas, datas,
valores monetários etc. As etiquetas seguem o esquema OntoNotes:

In [ ]:
for ent in documento.ents:
    print(f"{ent.text:<30} {ent.label_:<8} {spacy.explain(ent.label_)}")

> **Nota:** O modelo identifica "pandemia" como ORG (organização). Isso é um erro
> de predição — o NER não é perfeito. Em projetos reais, é comum ajustar o modelo
> com exemplos anotados (fine-tuning) ou usar regras de pós-processamento.

## 10. Stopwords

Stopwords são palavras de alta frequência e baixa carga semântica (artigos,
preposições, conjunções). Removê-las é um pré-processamento comum em classificação
e busca.

In [ ]:
# Quantas stopwords existem no modelo?
stops = list(nlp.Defaults.stop_words)
print(f"Total de stopwords: {len(stops)}")
print(f"Amostra: {stops[:20]}")

In [ ]:
# Adicionar uma stopword customizada
nlp.Defaults.stop_words.add("eita")
nlp.vocab['eita'].is_stop = True
print(nlp.vocab['eita'].is_stop)

In [ ]:
# Remover stopwords de um texto
tokens_sem_stop = [token.text for token in documento if not token.is_stop]
print("Original:", documento.text)
print("\nSem stopwords:", ' '.join(tokens_sem_stop))

## 11. Vocabulário e hashing

O spaCy armazena strings em uma tabela de hash para acesso rápido. Cada string
recebe um ID numérico único, e a tabela é bidirecional:

In [ ]:
# String → hash
hash_id = nlp.vocab.strings["dados"]
print(f"Hash de 'dados': {hash_id}")

# Hash → string
print(f"String do hash: {nlp.vocab.strings[hash_id]}")

In [ ]:
# Lexeme: entrada do vocabulário (compartilhada entre todos os docs)
lex = nlp.vocab["dados"]
print(f"texto={lex.text}, orth={lex.orth}, is_alpha={lex.is_alpha}, is_lower={lex.is_lower}")

## 12. Vetores de palavras e similaridade semântica

O modelo `lg` inclui vetores de 300 dimensões para ~500 mil palavras (word
embeddings). O vetor de um documento é a média dos vetores de seus tokens.

A similaridade é o **cosseno** entre vetores — valor entre 0 (ortogonais) e 1
(mesma direção).

In [ ]:
# Vetor de um documento (300 dimensões)
vec = nlp("dados são uma nova forma de ver o mundo").vector
print(f"Formato: {vec.shape}")
print(f"Primeiras 5 dimensões: {vec[:5]}")

In [ ]:
# Similaridade entre sentenças
doc1 = nlp("Ele viaja regularmente de carro")
doc2 = nlp("Ela regularmente de avião viaja")
print(f"Similaridade: {doc1.similarity(doc2):.4f}")

In [ ]:
# Similaridade entre tokens (parônimos)
doc3 = nlp("Devemos dizer comprimento ou cumprimento?")
t_a, t_b = doc3[2], doc3[4]
print(f"'{t_a}' vs '{t_b}': {t_a.similarity(t_b):.4f}")

In [ ]:
# Similaridade entre spans
doc4 = nlp("Ele pede descrição. Ele pede discrição")
span_a = doc4[0:3]  # Ele pede descrição
span_b = doc4[4:7]  # Ele pede discrição
print(f"'{span_a}' vs '{span_b}': {span_a.similarity(span_b):.4f}")

> **Limitação:** Esses vetores são word embeddings estáticos (word2vec). Eles não
> capturam contexto — "banco" (de sentar) e "banco" (instituição) têm o mesmo vetor.
> Modelos contextuais como BERT (que veremos em capítulos posteriores) resolvem isso.

## 13. Casamento de padrões com `Matcher`

O `Matcher` é uma引擎 de regras sobre sequências de tokens. Útil para extrair
informação estruturada que o NER perde: números de telefone, CNPJ, expressões
multi-palavra.

In [ ]:
from spacy.matcher import Matcher

# Encontrar números de telefone
doc5 = nlp("Você pode ligar para (51) - 9964656570 ou (11) 12344988")

matcher = Matcher(nlp.vocab)

# Padrão: '(' + dois dígitos + ')' + '-' opcional + dígitos
padrao_telefone = [
    {"ORTH": "("},
    {"SHAPE": "dd"},
    {"ORTH": ")"},
    {"ORTH": "-", "OP": "?"},
    {"IS_DIGIT": True}
]
matcher.add("telefone", [padrao_telefone])

matches = matcher(doc5)
for match_id, inicio, fim in matches:
    print(f"Encontrado: {doc5[inicio:fim]}")

In [ ]:
# Múltiplos padrões para variações de grafia
doc6 = nlp(
    "Estamos infectados com micro organismos. "
    "MICROORGANISMOS são perigosos. "
    "Não enxergamos micro-organismos"
)

matcher2 = Matcher(nlp.vocab)
matcher2.add("micro-organismos", [
    [{"LOWER": "micro-organismos"}],
    [{"LOWER": "microorganismos"}],
    [{"LOWER": "micro"}, {"LOWER": "organismos"}],
])

for _, inicio, fim in matcher2(doc6):
    print(f"Match: '{doc6[inicio:fim]}'")


> **Dica:** `{"OP": "?"}` torna o token opcional, `{"OP": "*"}` permite zero ou
> mais ocorrências, `{"OP": "+"}` exige uma ou mais. Isso dá flexibilidade para
> lidar com pontuação variável, espaçamento e tokens intermediários.

## 14. Visualização com displaCy

O `displaCy` renderiza anotações do spaCy como HTML interativo — excelente para
exploração, debugging e apresentações.

In [ ]:
from spacy import displacy

# Entidades nomeadas (estilo 'ent')
displacy.render(documento, style="ent", jupyter=True)

In [ ]:
# Árvore de dependências (estilo 'dep')
displacy.render(
    documento,
    style="dep",
    jupyter=True,
    options={
        'compact': True,
        'distance': 80,
        'color': '#FFFFFF',
        'bg': '#000000',
        'font': 'Arial'
    }
)

## 15. Gerenciamento do pipeline

Em produção, frequentemente removemos componentes que não são necessários para
reduzir latência e consumo de memória.

In [ ]:
print(f"Pipeline completo: {nlp.pipe_names}")

# Remover tok2vec (derruba parser e NER com ele)
nlp.remove_pipe('tok2vec')
print(f"Sem tok2vec:       {nlp.pipe_names}")

# Reinserir após morphologizer
nlp.add_pipe('tok2vec', after='morphologizer')
print(f"Restaurado:        {nlp.pipe_names}")

### Pipeline sob medida

Para tarefas de classificação de texto, você pode criar um pipeline mínimo:

In [ ]:
# Pipeline mínimo: só tokenização
nlp_light = spacy.blank('pt')
print(f"Pipeline blank: {nlp_light.pipe_names}")

# Adicionar apenas o que precisa
nlp_light.add_pipe('sentencizer')  # segmentação de sentenças (regras, sem ML)
print(f"Com sentencizer: {nlp_light.pipe_names}")

| Componente | Custo | Substituição leve |
|---|---|---|
| `tok2vec` + `parser` | alto | `sentencizer` (baseado em regras) |
| `tok2vec` + `ner` | alto | `Matcher` com lista de entidades |
| `lemmatizer` | médio | dispensável para classificação com TF-IDF |
| `tagger` / `morphologizer` | médio | necessário se você usa POS como feature |

## 16. Resumo e próximos passos

Este capítulo cobriu o essencial do spaCy para PLN em português:

| Funcionalidade | API principal |
|---|---|
| Tokenização | `token.text`, índices, spans |
| Atributos lexicais | `token.is_stop`, `is_alpha`, `is_punct`, `shape_` |
| POS tagging | `token.pos_`, `token.tag_`, `spacy.explain()` |
| Morfologia | `token.morph` |
| Lematização | `token.lemma_` |
| Dependências | `token.dep_`, `token.head` |
| NER | `doc.ents`, `ent.label_` |
| Stopwords | `nlp.Defaults.stop_words` |
| Vocabulário | `nlp.vocab.strings`, `nlp.vocab['palavra']` |
| Similaridade | `doc1.similarity(doc2)` |
| Matcher | `Matcher(nlp.vocab)`, padrões de tokens |
| Visualização | `displacy.render(doc, style='ent'|'dep')` |
| Pipeline | `nlp.pipe_names`, `remove_pipe`, `add_pipe` |

No próximo capítulo, exploraremos o **NLTK** — a outra grande biblioteca de PLN
do ecossistema Python — com foco em stemming, tokenização, frequência e tagging.